# Silver Layer — Data Cleaning & Transformation

**Medallion Stage:** Silver (Cleaned, Enriched)

This notebook transforms the bronze layer into a clean, type-safe, feature-enriched
DataFrame ready for analytics. Every transformation is documented with its rationale.

**Decisions made here:**
- Remove three zero-variance columns (EmployeeCount, Over18, StandardHours)
- Encode binary categoricals to 0/1 for downstream numeric operations
- Cast high-cardinality strings to `category` dtype (memory + speed)
- Add ordinal labels for human-readable reporting
- Engineer 7 derived features capturing relationships the raw columns don't express

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('Set2')

df = pd.read_csv(ROOT / 'data' / 'bronze' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Bronze loaded: {df.shape}')

## 1. Remove Constant Columns

These three columns have a single unique value across all 1,470 rows and carry zero information.

In [ ]:
CONSTANTS = ['EmployeeCount', 'Over18', 'StandardHours']
print('Values before drop:')
for col in CONSTANTS:
    print(f'  {col}: {df[col].unique()}')

df = df.drop(columns=CONSTANTS)
print(f'\nShape after drop: {df.shape}')

## 2. Binary Encoding

| Column | Before | After | Rule |
|---|---|---|---|
| Attrition | Yes / No | 1 / 0 | Industry standard for binary target |
| OverTime | Yes / No | 1 / 0 | Enables numeric aggregation |
| Gender | Male / Female | 1 / 0 | 1 = Male |

Note: Gender encoding is for ML compatibility only. All reporting uses the original labels.

In [ ]:
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)
df['OverTime']  = (df['OverTime']  == 'Yes').astype(int)
df['Gender']    = (df['Gender']    == 'Male').astype(int)

print('Binary columns encoded:')
for col in ['Attrition', 'OverTime', 'Gender']:
    print(f'  {col}: {df[col].value_counts().to_dict()}')

## 3. Type Casting — Categoricals

In [ ]:
CAT_COLS = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

before_mem = df.memory_usage(deep=True).sum() / 1024
for col in CAT_COLS:
    df[col] = df[col].astype('category')
after_mem = df.memory_usage(deep=True).sum() / 1024

print(f'Memory: {before_mem:.1f} KB  →  {after_mem:.1f} KB  (saved {before_mem - after_mem:.1f} KB)')

## 4. Ordinal Labels

The dataset uses integer codes (1–4 or 1–5) for satisfaction and education fields.
We add human-readable label columns alongside the numeric codes.

In [ ]:
ORDINAL_MAPPINGS = {
    'Education':                {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'EnvironmentSatisfaction':  {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobInvolvement':           {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobSatisfaction':          {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'PerformanceRating':        {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'},
    'RelationshipSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance':          {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
}

for col, mapping in ORDINAL_MAPPINGS.items():
    df[f'{col}_Label'] = df[col].map(mapping)

print(f'Label columns added: {[c for c in df.columns if c.endswith("_Label")]}')

## 5. Feature Engineering

Seven derived features — each capturing a relationship invisible in the raw columns.

In [ ]:
# 1. Composite satisfaction score across all 4 dimensions
sat_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']
df['SatisfactionComposite'] = df[sat_cols].mean(axis=1).round(2)

# 2. Average years per employer — signals loyalty vs. job-hopping
df['YearsPerCompany'] = np.where(
    df['NumCompaniesWorked'] > 0,
    (df['TotalWorkingYears'] / df['NumCompaniesWorked']).round(2),
    df['TotalWorkingYears']
)

# 3. Income relative to job-level median — captures salary compression
level_median = df.groupby('JobLevel')['MonthlyIncome'].transform('median')
df['IncomeVsLevelMedian'] = ((df['MonthlyIncome'] - level_median) / level_median).round(4)

# 4. Promotion stagnation ratio — years since promotion / total tenure
df['PromotionLag'] = (df['YearsSinceLastPromotion'] / df['YearsAtCompany'].clip(lower=1)).round(4)

# 5. Manager tenure ratio — how long the current manager has been in place
df['ManagerTenureRatio'] = (df['YearsWithCurrManager'] / df['YearsAtCompany'].clip(lower=1)).round(4)

# 6. Distance tier — buckets commute distance into qualitative bands
df['DistanceTier'] = pd.cut(
    df['DistanceFromHome'],
    bins=[0, 5, 10, 20, 100],
    labels=['Near', 'Moderate', 'Far', 'Very Far'],
)

# 7. Age band
df['AgeBand'] = pd.cut(
    df['Age'],
    bins=[17, 25, 35, 45, 55, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '55+'],
)

print(f'Shape after feature engineering: {df.shape}')
print('New columns:', ['SatisfactionComposite', 'YearsPerCompany', 'IncomeVsLevelMedian',
                       'PromotionLag', 'ManagerTenureRatio', 'DistanceTier', 'AgeBand'])

## 6. Distribution of Derived Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

plots = [
    ('SatisfactionComposite', 'Satisfaction Composite'),
    ('YearsPerCompany',        'Avg Years per Employer'),
    ('IncomeVsLevelMedian',    'Income vs Level Median'),
    ('PromotionLag',           'Promotion Stagnation Ratio'),
    ('ManagerTenureRatio',     'Manager Tenure Ratio'),
]

for i, (col, title) in enumerate(plots):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].legend(fontsize=9)

axes[5].set_visible(False)
plt.suptitle('Distribution of Engineered Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Validate & Save Silver Layer

In [ ]:
from src.utils.config import load_config
from src.silver.transform import SilverTransformer

cfg = load_config()
bronze_raw = pd.read_csv(ROOT / 'data' / 'bronze' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
transformer = SilverTransformer(cfg)
silver_df = transformer.transform(bronze_raw)

silver_path = ROOT / 'data' / 'silver' / 'employee_attrition_silver.parquet'
print(f'Silver parquet written: {silver_path}')
print(f'Final shape: {silver_df.shape}')
print(f'\nAttrition rate (silver): {silver_df["Attrition"].mean():.1%}')
print(f'Null count: {silver_df.drop(columns=["_silver_ts"]).isnull().sum().sum()}')

---
## Silver Layer Summary

| Step | Action | Columns Affected |
|---|---|---|
| Remove constants | Dropped 3 zero-variance cols | -3 columns |
| Binary encoding | Yes/No → 1/0 | Attrition, OverTime, Gender |
| Category casting | object → category | 5 columns |
| Ordinal labels | Added readable labels | +7 label columns |
| Feature engineering | Computed derived metrics | +7 feature columns |
| Metadata | Added `_silver_ts` timestamp | +1 column |

**Next step:** `03_gold_feature_engineering.ipynb`